# `prepare_blm` の形と符号の確認

Beamのpacked $(T,E,B)$ 係数を、Convolutionで使用する $(0,+2,-2)$、$n=-n_{\max},\ldots,n_{\max}$ の形へ変換します。符号、spinの交換、最後の複素共役を既知係数で確認します。

In [ ]:
using EBCoDeCMB
using Test

## 1. $n=1$ と $n=2$ の入力を作る

奇数と偶数の両方を入れることで $(-1)^n$ の符号を確認します。

In [ ]:
cp = ConvolutionParams(
    nside_output=2, lstart=0, lstop=4, lmax=4,
    bnmax=2, numofbeam=1,
)
packed_index(l, n, lmax) =
    n * (lmax + 1) - n * (n - 1) ÷ 2 + (l - n) + 1

coefficient_count = (cp.lmax + 1) * (cp.lmax + 2) ÷ 2
blm_teb = zeros(ComplexF64, 3, coefficient_count)
l = 3
for n in 1:2
    index = packed_index(l, n, cp.lmax)
    blm_teb[:, index] .= ComplexF64[
        n + (n + 1)im, 2n - 0.25im, -n + (3n - 1)im,
    ]
end
blm_teb

## 2. 出力の構造

`blm_slice(prepared, l)` は `(spin0, spin+2, spin-2)` を返します。各blockの行は $n=-n_{\max}:n_{\max}$、列はbeamです。

In [ ]:
prepared = prepare_blm(cp, blm_teb)
blocks = blm_slice(prepared, l)
nmax = calc_nmax(cp, l)
(
    prepared=prepared,
    spin_order=(0, +2, -2),
    n_order=collect(-nmax:nmax),
    block_sizes=size.(blocks),
)

In [ ]:
spin0, spin_plus2, spin_minus2 = blocks
[
    (n=n, spin0=spin0[n+nmax+1, 1],
     spin_plus2=spin_plus2[n+nmax+1, 1],
     spin_minus2=spin_minus2[n+nmax+1, 1])
    for n in -nmax:nmax
]

## 3. 正の $n$、負の $n$、複素共役を確認

正の $n$ には入力spin係数の複素共役が保存されます。負のmodeは $B_s(l,-n)=(-1)^n B_{-s}(l,n)^*$ を満たします。

In [ ]:
for n in 1:2
    index = packed_index(l, n, cp.lmax)
    raw_spin = (
        blm_teb[1, index],
        -(blm_teb[2, index] + im * blm_teb[3, index]),
        -(blm_teb[2, index] - im * blm_teb[3, index]),
    )
    positive = n + nmax + 1
    negative = -n + nmax + 1
    phase = (-1)^n

    @test spin0[positive, 1] == conj(raw_spin[1])
    @test spin_plus2[positive, 1] == conj(raw_spin[2])
    @test spin_minus2[positive, 1] == conj(raw_spin[3])
    @test spin0[negative, 1] == phase * raw_spin[1]
    @test spin_plus2[negative, 1] == phase * raw_spin[3]
    @test spin_minus2[negative, 1] == phase * raw_spin[2]
end
"odd/even n and conjugation checks passed"

## 4. 複数beam

複数beamでは各blockの第2軸がbeamになります。

In [ ]:
cp_two_beams = ConvolutionParams(cp; numofbeam=2)
prepared_two = prepare_blm(
    cp_two_beams, [blm_teb, 2 .* blm_teb]; precision=Float32,
)
(
    prepared=prepared_two,
    element_type=eltype(prepared_two),
    l3_block_sizes=size.(blm_slice(prepared_two, 3)),
)